# Capítulo 11 — Álgebra lineal como lenguaje de modelos

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## ¿Qué significa que un sistema esté mal condicionado?

Dos sistemas 2x2 con soluciones idénticas: uno bien condicionado y otro casi
singular. Se perturba el término independiente un 1 % y se mira qué pasa.

La figura responde: ¿por qué una matriz puede amplificar tus errores, y cuánto?

Ejecutar:  python fig_condicionamiento.py

*(script original: `codigo/fig_condicionamiento.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(31)

CASOS = [
    ("Bien condicionado", np.array([[1.0, 0.2], [0.2, 1.0]])),
    ("Casi singular", np.array([[1.0, 0.999], [0.999, 1.0]])),
]
x_real = np.array([1.0, 1.0])

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.4))

for ax, (titulo, A) in zip(axes, CASOS):
    b = A @ x_real
    kappa = np.linalg.cond(A)

    # Las dos rectas del sistema
    xx = np.linspace(-3, 5, 200)
    for i, color in zip(range(2), [C.blue, C.red]):
        ax.plot(xx, (b[i] - A[i, 0] * xx) / A[i, 1], color=color, lw=1.8,
                alpha=0.9)

    # 300 perturbaciones del 1 % en b
    soluciones = []
    for _ in range(300):
        bp = b * (1 + 0.01 * r.standard_normal(2))
        soluciones.append(np.linalg.solve(A, bp))
    soluciones = np.array(soluciones)
    ax.plot(soluciones[:, 0], soluciones[:, 1], ".", color=C.ochre, ms=3,
            alpha=0.6)
    ax.plot(*x_real, "*", color=C.ink, ms=15, zorder=6)

    dispersión = np.linalg.norm(soluciones - x_real, axis=1).std()
    ax.set_title(f"{titulo}\n$\\kappa$ = {kappa:.1f},  dispersión = "
                 f"{dispersión:.2f}", fontsize=10)
    ax.set_xlabel("$x_1$"), ax.set_ylabel("$x_2$")
    ax.set_xlim(-2, 4), ax.set_ylim(-2, 4)
    ax.set_aspect("equal")
    print(f"{titulo:22s} kappa = {kappa:8.1f}   dispersión = {dispersión:.3f}")

axes[1].annotate("las dos rectas son\ncasi paralelas",
                 xy=(0.5, 1.5), xytext=(-1.7, 3.2), fontsize=8.6, color=C.red,
                 arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Qué es un autovector, físicamente?

Tres masas unidas por muelles: se calculan los modos normales y se muestra que
cualquier movimiento es una superposición de ellos.

La figura responde: ¿por qué diagonalizar una matriz es elegir el punto de
vista en el que el problema se desacopla?

Ejecutar:  python fig_modos_normales.py

*(script original: `codigo/fig_modos_normales.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

N = 3
K = np.array([[2.0, -1.0, 0.0], [-1.0, 2.0, -1.0], [0.0, -1.0, 2.0]])
w2, V = np.linalg.eigh(K)
w = np.sqrt(w2)

fig, axes = plt.subplots(2, 3, figsize=(11.2, 5.4),
                         gridspec_kw={"hspace": 0.45})

# --- Fila 1: los tres modos ----------------------------------------------
for k in range(N):
    ax = axes[0, k]
    x = np.arange(1, N + 1)
    ax.axhline(0, color=C.grey, lw=1.0)
    for i in range(N):
        ax.plot([x[i], x[i]], [0, V[i, k]], color=C.ink, lw=1.0, alpha=0.5)
    ax.plot(x, V[:, k], "o-", color=[C.blue, C.green, C.red][k], ms=11, lw=2)
    ax.set_ylim(-0.9, 0.9)
    ax.set_xticks(x), ax.set_xticklabels(["$m_1$", "$m_2$", "$m_3$"])
    ax.set_title(f"Modo {k+1}: $\\omega$ = {w[k]:.3f}", fontsize=10)
    if k == 0:
        ax.set_ylabel("amplitud")

# --- Fila 2: una condición inicial arbitraria y su descomposición --------
t = np.linspace(0, 30, 1500)
x0 = np.array([1.0, 0.0, 0.0])          # sólo la primera masa desplazada
coef = V.T @ x0                          # proyección sobre los modos
trayectoria = np.array([sum(coef[k] * np.cos(w[k] * ti) * V[:, k]
                            for k in range(N)) for ti in t])

ax = axes[1, 0]
for i in range(N):
    ax.plot(t, trayectoria[:, i], lw=1.2, label=f"$m_{i+1}$")
ax.set_xlabel("tiempo"), ax.set_ylabel("desplazamiento")
ax.set_title("Movimiento real: complicado", fontsize=10)
ax.legend(fontsize=7.6, ncol=3)

ax = axes[1, 1]
for k in range(N):
    ax.plot(t, coef[k] * np.cos(w[k] * t), lw=1.4,
            color=[C.blue, C.green, C.red][k], label=f"modo {k+1}")
ax.set_xlabel("tiempo"), ax.set_ylabel("amplitud del modo")
ax.set_title("En la base de modos: tres cosenos", fontsize=10)
ax.legend(fontsize=7.6, ncol=3)

ax = axes[1, 2]
ax.bar(np.arange(1, N + 1), coef**2 / (coef**2).sum(),
       color=[C.blue, C.green, C.red], width=0.6)
ax.set_xticks(np.arange(1, N + 1))
ax.set_xlabel("modo"), ax.set_ylabel("fracción de energía")
ax.set_title("Cuánto pesa cada modo", fontsize=10)

print("frecuencias:", np.round(w, 4))
print("coeficientes:", np.round(coef, 4))
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Todos los autovalores estables y el sistema crece igualmente. ¿Cómo?

Crecimiento transitorio en una matriz no normal: la energía se multiplica por
mil antes de decaer, aunque los dos autovalores sean negativos.

La figura responde: ¿basta con mirar los autovalores para saber si un sistema
es estable en la práctica?

Ejecutar:  python fig_no_normal.py

*(script original: `codigo/fig_no_normal.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

# Matriz no normal: autovalores -1 y -2, pero autovectores casi paralelos
A = np.array([[-1.0, 200.0], [0.0, -2.0]])
lam = np.linalg.eigvals(A)

t = np.linspace(0, 12, 2000)
crecimiento = np.array([np.linalg.norm(expm(A * ti), 2) for ti in t])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

ax1.semilogy(t, crecimiento, color=C.red, lw=2.0,
             label=r"$\|e^{At}\|$ real")
ax1.semilogy(t, np.exp(np.max(lam.real) * t), "--", color=C.blue, lw=1.8,
             label=r"$e^{\lambda_{\max}t}$ (lo que dicen los autovalores)")
ax1.axhline(1, color=C.grey, lw=1.0)
ax1.set_xlabel("tiempo"), ax1.set_ylabel("amplificación")
ax1.set_title("Autovalores $-1$ y $-2$: ambos estables")
ax1.legend(fontsize=8)
ax1.annotate(f"amplificación máxima: ×{crecimiento.max():.0f}",
             xy=(t[np.argmax(crecimiento)], crecimiento.max()),
             xytext=(3.5, 3), fontsize=8.6, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))

# --- Trayectorias en el plano --------------------------------------------
for ang in np.linspace(0, np.pi, 9)[:-1]:
    x0 = np.array([np.cos(ang), np.sin(ang)])
    tr = np.array([expm(A * ti) @ x0 for ti in np.linspace(0, 6, 400)])
    ax2.plot(tr[:, 0], tr[:, 1], color=C.blue, lw=1.0, alpha=0.8)
    ax2.plot(*x0, "o", color=C.ink, ms=3)
th = np.linspace(0, 2 * np.pi, 200)
ax2.plot(np.cos(th), np.sin(th), "--", color=C.grey, lw=1.2)
ax2.set_xlabel("$x_1$"), ax2.set_ylabel("$x_2$")
ax2.set_title("Salen del círculo unidad antes de volver")
ax2.set_xlim(-60, 60), ax2.set_ylim(-1.3, 1.3)

print(f"autovalores: {lam}")
print(f"amplificación transitoria máxima: {crecimiento.max():.1f}")
print(f"número de condición de los autovectores: "
      f"{np.linalg.cond(np.linalg.eig(A)[1]):.1f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## SVD: la descomposición que explica compresión, ajuste y PCA a la vez.

Se aplica a una imagen sintética: espectro de valores singulares y
reconstrucciones con distintos rangos.

La figura responde: ¿cuánta información hay realmente en una matriz?

Ejecutar:  python fig_svd.py

*(script original: `codigo/fig_svd.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(11)

n = 200
y, x = np.mgrid[0:n, 0:n] / n
imagen = (np.sin(6 * np.pi * x) * np.cos(4 * np.pi * y)
          + 1.5 * np.exp(-((x - 0.3)**2 + (y - 0.7)**2) / 0.02)
          + 0.8 * (np.abs(x - y) < 0.05)
          + 0.05 * r.standard_normal((n, n)))

U, S, Vt = np.linalg.svd(imagen, full_matrices=False)

fig = plt.figure(figsize=(11.2, 5.0))
gs = fig.add_gridspec(2, 4, width_ratios=[1.3, 1, 1, 1], wspace=0.3, hspace=0.35)

ax = fig.add_subplot(gs[:, 0])
ax.semilogy(S, "o-", color=C.blue, ms=3, lw=1.0)
ax.set_xlabel("índice $k$"), ax.set_ylabel("valor singular $\\sigma_k$")
ax.set_title("Espectro de valores singulares")
ax.axhline(S[0] * 1e-2, color=C.grey, ls="--", lw=1.0)
ax.text(60, S[0] * 1.3e-2, "1 % del mayor", fontsize=8, color=C.grey)
ax.set_xlim(0, 200)

RANGOS = [1, 5, 20, 200]
for j, k in enumerate(RANGOS):
    ax = fig.add_subplot(gs[j // 2, 1 + (j % 2)])
    aprox = (U[:, :k] * S[:k]) @ Vt[:k]
    ax.imshow(aprox, cmap="gray", interpolation="nearest")
    ax.set_xticks([]), ax.set_yticks([]), ax.grid(False)
    guardado = 100 * (1 - k * (2 * n + 1) / n**2)
    ax.set_title(f"rango {k}  ({guardado:.0f} % menos datos)"
                 if k < n else "original (rango completo)", fontsize=9)

ax = fig.add_subplot(gs[0, 3])
energia = np.cumsum(S**2) / np.sum(S**2)
ax.plot(np.arange(1, len(S) + 1), energia, color=C.red, lw=1.8)
ax.axhline(0.99, color=C.grey, ls="--", lw=1.0)
k99 = int(np.searchsorted(energia, 0.99) + 1)
ax.axvline(k99, color=C.grey, ls="--", lw=1.0)
ax.text(k99 + 4, 0.5, f"99 % con\nrango {k99}", fontsize=8, color=C.ink)
ax.set_xlabel("rango $k$"), ax.set_ylabel("energía acumulada")
ax.set_title("Cuánto captura cada rango", fontsize=9)
ax.set_xlim(0, 120)

print(f"rango 99 % de energía: {k99} de {n}")
print(f"sigma_1/sigma_200 = {S[0]/S[-1]:.1f}  (número de condición)")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
